In [1]:
"""
IMA(1,1) and ARIMA(1,1,1) Forecasting Models
===============================================
Real-time recursive forecasts of log real TTF NG prices.
IMA(1,1):     Δr_t = ε_t + θ_1·ε_{t-1}
ARIMA(1,1,1): Δr_t = φ_1·Δr_{t-1} + ε_t + θ_1·ε_{t-1}
MA term active at h=1 only. Level recovered by cumulating Δr_hat.
"""

import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings("ignore")

In [2]:
# ── Parameters ────────────────────────────────────────────────────────────────
HORIZONS    = [1, 3, 6, 9, 12, 15, 18, 21, 24]
EVAL_START  = "2015-01-01"
INPUT_FILE  = "Input_TTF_NG_Real_Average_Prices.xlsx"
OUTPUT_FILE = "Output_IMA_ARIMA_forecasts.xlsx"

In [3]:
# Model specifications: (label, ARIMA order)
SPECIFICATIONS = [
    ("IMA(1,1)",     (0, 1, 1)),
    ("ARIMA(1,1,1)", (1, 1, 1)),
]

In [4]:
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_excel(INPUT_FILE, parse_dates=["date"])
df = df[["date","price_real"]].sort_values("date").reset_index(drop=True)
df["log_price"] = np.log(df["price_real"])

In [5]:
# ── Helper: actual real price for a given year-month ─────────────────────────
def get_actual(ym_str):
    m = df[df["date"].dt.to_period("M").astype(str) == ym_str]
    return m["price_real"].values[0] if len(m) == 1 else np.nan

# ── Helper: forecasts for a given ARIMA order ────────────────────────────────
def arima_forecasts(history, order, horizons):
    model = ARIMA(history, order=order).fit()
    h_max = max(horizons)
    # forecast() returns predicted log levels (statsmodels undoes differencing)
    f_log = model.forecast(steps=h_max)
    return {h: f_log[h-1] for h in horizons}

In [6]:
# ── Main forecasting loop ─────────────────────────────────────────────────────
records = []
origins = df[df["date"] >= EVAL_START]["date"].tolist()

for origin_date in origins:
    history = df[df["date"] <= origin_date]["log_price"].values

    # Need at least 3 observations
    if len(history) < 3:
        continue

    for label, order in SPECIFICATIONS:
        try:
            forecasts = arima_forecasts(history, order, HORIZONS)
        except Exception:
            continue

        for h in HORIZONS:
            actual_ym      = (origin_date + pd.DateOffset(months=h)).strftime("%Y-%m")
            forecast_level = np.exp(forecasts[h])
            actual_val     = get_actual(actual_ym)

            records.append({
                "forecast_origin": origin_date.strftime("%Y-%m-%d"),
                "horizon":         h,
                "model":           label,
                "actual_month":    actual_ym,
                "forecast":        forecast_level,
                "actual":          actual_val,
            })

In [7]:
# ── Save output ───────────────────────────────────────────────────────────────
results = pd.DataFrame(records)
results.to_excel(OUTPUT_FILE, index=False)

In [8]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("IMA(1,1) and ARIMA(1,1,1) forecasting complete.")
print(f"  Models:           {[s[0] for s in SPECIFICATIONS]}")
print(f"  Forecast origins: {results['forecast_origin'].nunique()}")
print(f"  Horizons:         {HORIZONS}")
print(f"  Total rows:       {len(results)}")
print(f"  Output saved to:  {OUTPUT_FILE}")
print()

IMA(1,1) and ARIMA(1,1,1) forecasting complete.
  Models:           ['IMA(1,1)', 'ARIMA(1,1,1)']
  Forecast origins: 132
  Horizons:         [1, 3, 6, 9, 12, 15, 18, 21, 24]
  Total rows:       2376
  Output saved to:  Output_IMA_ARIMA_forecasts.xlsx



In [9]:
# Sample — first origin, both models
first_origin = results["forecast_origin"].min()
sample = results[results["forecast_origin"] == first_origin]
print(f"Sample — first origin ({first_origin}):")
print(sample[["model","horizon","actual_month","forecast","actual"]].to_string(index=False))

Sample — first origin (2015-01-31):
       model  horizon actual_month  forecast    actual
    IMA(1,1)        1      2015-02 19.766375 22.938516
    IMA(1,1)        3      2015-04 19.766375 22.046423
    IMA(1,1)        6      2015-07 19.766375 20.679393
    IMA(1,1)        9      2015-10 19.766375 18.160884
    IMA(1,1)       12      2016-01 19.766375 13.882111
    IMA(1,1)       15      2016-04 19.766375 12.101522
    IMA(1,1)       18      2016-07 19.766375 14.115551
    IMA(1,1)       21      2016-10 19.766375 15.958961
    IMA(1,1)       24      2017-01 19.766375 19.873384
ARIMA(1,1,1)        1      2015-02 19.634558 22.938516
ARIMA(1,1,1)        3      2015-04 19.637123 22.046423
ARIMA(1,1,1)        6      2015-07 19.762080 20.679393
ARIMA(1,1,1)        9      2015-10 19.644245 18.160884
ARIMA(1,1,1)       12      2016-01 19.755325 13.882111
ARIMA(1,1,1)       15      2016-04 19.650578 12.101522
ARIMA(1,1,1)       18      2016-07 19.749322 14.115551
ARIMA(1,1,1)       21      20